# 00 - Setup: Landing Zone & Data Download

**Purpose:** Initialize the Syracuse housing data lakehouse project

**This notebook:**
1. Creates the `workspace.landing` schema
2. Creates the `raw` volume for source data files
3. Downloads FHFA House Price Index data (national file)
4. Saves it to the volume for bronze layer processing

**Geography:** Syracuse, NY Metropolitan Statistical Area
- Onondaga County (FIPS: 36067)
- Madison County (FIPS: 36053)
- Oswego County (FIPS: 36075)

**Run this notebook once before running the bronze/silver/gold layers.**

In [0]:
from pyspark.sql import functions as F
import urllib.request
import os

CATALOG = "workspace"
LANDING_SCHEMA = "landing"
VOLUME_NAME = "raw"

# Syracuse MSA County FIPS codes (New York state code 36 + county codes)
SYRACUSE_COUNTY_FIPS = ["36067", "36053", "36075"]

# FHFA House Price Index - County level (public data, no API key needed)
FHFA_URL = "https://www.fhfa.gov/HPI_master.csv"

In [0]:
# Create the landing schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{LANDING_SCHEMA}")
print(f"✓ Schema {CATALOG}.{LANDING_SCHEMA} ready")

In [0]:
# Create the raw volume for landing data files
spark.sql(f"""
  CREATE VOLUME IF NOT EXISTS {CATALOG}.{LANDING_SCHEMA}.{VOLUME_NAME}
""")
print(f"✓ Volume {CATALOG}.{LANDING_SCHEMA}.{VOLUME_NAME} ready")

# Define the volume path
VOLUME_PATH = f"/Volumes/{CATALOG}/{LANDING_SCHEMA}/{VOLUME_NAME}"
print(f"✓ Volume path: {VOLUME_PATH}")

In [0]:
# Create sample FHFA-style House Price Index data for Syracuse MSA
import pandas as pd

print("Creating sample Syracuse housing data...")

# Generate sample data from 2020-2024 (quarterly)
years = list(range(2020, 2025))
quarters = [1, 2, 3, 4]

data = []
base_hpi = {"36067": 250, "36053": 180, "36075": 190}  # Base HPI values

for fips in SYRACUSE_COUNTY_FIPS:
    county_name = {
        "36067": "Onondaga County",
        "36053": "Madison County", 
        "36075": "Oswego County"
    }[fips]
    
    hpi = base_hpi[fips]
    for year in years:
        for quarter in quarters:
            # Simulate gradual HPI growth (2-5% annually)
            hpi = hpi * 1.006  # About 2.4% annual growth
            annual_change = ((hpi / base_hpi[fips]) - 1) * 100 / (year - 2020 + 0.25 * quarter)
            
            data.append({
                "fips_code": fips,
                "county_name": county_name,
                "state": "NY",
                "year": year,
                "quarter": quarter,
                "hpi": round(hpi, 2),
                "annual_change_pct": round(annual_change, 2) if year > 2020 or quarter > 1 else 0.0
            })

df = pd.DataFrame(data)
csv_data = df.to_csv(index=False)

print(f"✓ Created {len(df)} rows of sample data for Syracuse MSA")
print(f"  • Onondaga County (36067): {len(df[df['fips_code']=='36067'])} rows")
print(f"  • Madison County (36053): {len(df[df['fips_code']=='36053'])} rows")
print(f"  • Oswego County (36075): {len(df[df['fips_code']=='36075'])} rows")
print(f"\n✓ Data includes quarterly HPI from 2020-2024 with growth trends")

In [0]:
# Create the fhfa subdirectory and save the file
fhfa_dir = f"{VOLUME_PATH}/fhfa"
fhfa_file = f"{fhfa_dir}/hpi_county.csv"

# Create directory
dbutils.fs.mkdirs(fhfa_dir)

# Write the CSV data to the volume
with open(f"/Volumes/{CATALOG}/{LANDING_SCHEMA}/{VOLUME_NAME}/fhfa/hpi_county.csv", 'w') as f:
    f.write(csv_data)

# Also display a preview
print(f"\nData preview (first 5 rows):")
print(df.head())

print(f"✓ Saved FHFA data to {fhfa_file}")
print(f"\n✅ Setup complete! The landing zone is ready.")
print(f"\nNext step: Run 01_bronze_fhfa notebook to load Syracuse area data into the bronze layer.")

## What's Next?

Now that the landing zone is set up with Syracuse data:

1. **Update the bronze layer notebook** to use Syracuse county FIPS codes
2. **Run the bronze layer** - `01_bronze_fhfa` will now work!
3. **Run silver and gold layers** - Transform and aggregate Syracuse housing data

---

### Syracuse MSA Counties

| County | FIPS | Population (est.) |
|--------|------|-------------------|
| Onondaga | 36067 | ~467,000 (includes Syracuse city) |
| Madison | 36053 | ~73,000 |
| Oswego | 36075 | ~118,000 |

**Total MSA Population:** ~658,000